# Databricks AutoML

## Introduction
Databricks AutoML allows you to automatically generate baseline models for classification, regression, and forecasting problems. It performs data preparation, model training, and hyperparameter tuning, and logs everything to MLflow.

In this notebook, we will prepare a dataset for a **Forecasting** task and learn how to launch an AutoML experiment via the UI.

**Note - as of now this feature does not work. Until resolved by Datarbricks, Forecasting AutoML job can be run only using classic compute cluster, which is not available in Databricks Free**

## 1. Prepare Data

AutoML in Databricks typically works best with Delta Tables. We will generate a synthetic time-series dataset representing daily sales and save it as a table.

In [0]:
from pyspark.sql.functions import col, to_date, lit

# Path to the dataset
file_path = "/databricks-datasets/bikeSharing/data-001/day.csv"

# Load the data
# The time column for AutoML Forecasting must be a timestamp or date
# The 'dteday' column is a string, so we'll convert it to a date
spark_df = (spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(file_path)
            .withColumn("date", to_date(col("dteday")))
            .withColumn("series_id", lit(1)) # Add a dummy identifier for the time series
            .select("date", col("cnt").alias("count"), "series_id")
           )

display(spark_df)

## 2. Save as Table

We will save this dataframe as a managed Delta table named `daily_bike_rentals`.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ai_ml_in_practice.automl_forecasting")
spark_df.write.mode("overwrite").saveAsTable("ai_ml_in_practice.automl_forecasting.daily_bike_rentals")
print("Table 'daily_bike_rentals' created successfully.")

## 3. Running AutoML

Now that the data is ready, follow these steps to run AutoML:

1.  Click on **Experiments** on the left sidebar -> **Forecasting** on the upper bar
2.  **Configure the experiment**:
    * **Training data**: Browse ai_ml_in_practice.automl_forecasting schema and select `daily_bike_rentals` table
    * **Time column**: Select `date`
    * **Forecast frequency**: Select `Daily`
    * **Forecast Horizon**: Enter `30` (to predict next 30 days)
    * **Target column**: Select `count`
    * **Prediction data path**: Browse and select `ai_ml_in_practice.automl_forecasting`
    * **Table name**: Enter `forecast_predictions`
    * **Register to location**: Browse and select `ai_ml_in_practice.automl_forecasting`
    * **Model name**: Enter `forecasting_automl`
    * Under **Advanced** tab:
        * **Experiment name**: Enter `count_daily_bike_rentals`
        * **Experiment directory**: Leave the default
        * **Time series identifier columns**: Select `series_id`
        * **Primary metric**: Select `RMSE`
        * **Preferred training frameworks**: Leave the default
        * **Holiday region**: Select `US - United States`
        * **Timeout (minutes)**: Set to `5`
4.  Click **Start training**.

Databricks will now train multiple models (Prophet, ARIMA, etc.), tune them, and rank them by the evaluation metric (RMSE). Once finished, you can click on the best run to see the generated notebook code.